# `ratio_study` - 03 Reporting

Automated PDF, XLSX, CSV and PNG output.

| Notebook | Covers |
|---|---|
| [01 Basics](01_basics.ipynb) | Sections 1-2: indicators, confidence intervals, compliance checks, IQR trimming, charts, comparisons |
| [02 Spatial](02_spatial.ipynb) | Section 3: ratio maps, Moran's I, hot/cold spots, the VEM map |
| [03 Reporting](03_reporting.ipynb) | Section 4: PDF / XLSX / CSV / PNG output |
| [04 Advanced diagnostics](04_advanced_diagnostics.ipynb) | Sections 5 and 7: ML / SHAP inequity diagnostics and sales-chasing detection |
| [05 Vertical equity](05_vertical_equity.ipynb) | Section 6: VEI, the Vertical Equity Panel, VEM, the Gini family |

All five use the anonymised sample dataset bundled with the package - no data preparation required.
Section numbers are kept from the original single-notebook tour so cross-references still line up.

## Setup

PDF output needs only the core install. Section 4.2 shows chart families; the ML page of the report needs the `ml` extra (see notebook 04).

In [ ]:
# Uncomment on first run.
# %pip install "ratio_study[ml]"

In [ ]:
import ratio_study as rs
import pandas as pd
import numpy as np
from pathlib import Path

# inline plotting must be enabled AFTER importing ratio_study
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

sample_df = rs.load_sample_dataset()

MARKET_VALUE = 'time_adjusted_sale_price'   # the market value proxy
ASSESSMENT   = 'assessment_value_new'       # the value under test (current CAMA model)

sample_df['asr'] = sample_df[ASSESSMENT] / sample_df[MARKET_VALUE]   # ratio column, needed by the spatial and VE functions

results_directory = Path().resolve()   # outputs are written here
print('results will be written to:', results_directory)

print('ratio_study version:', rs.__version__)
print(f'{len(sample_df):,} sales, {sample_df.shape[1]} columns')

---
# 4. Automated reporting

The Standard sets out what a published ratio study owes its reader. The report should state
the purpose, identify data sources, and describe the methodology — sales time frame, data
edits, time trends, outlier trimming, statistics used, and the proper use of results — and
the exhibits should carry, for each stratum: the tax year, parcel and sale
counts, the number trimmed, level, uniformity, price-related bias, and confidence
intervals.

`ratios_report()` is `ratios()` plus publication: same arguments, same indicators, same
compliance commentary — and a finished PDF.

| Format | Argument | Contents |
|---|---|---|
| **PDF** | on by default | Ratio tables, compliance comments, charts, vertical equity panel |
| **XLSX** | `export_ratios='xlsx'` | One tab per grouping level |
| **CSV** | `export_ratios='csv'` | Flat table |
| **Both** | `export_ratios='both'` | XLSX and CSV |
| **ZIP / GZIP** | `export_in_compacted_file='zip'` | Everything above in one archive |
| **PNG** | `graph_filepath=` on any chart | Individual charts, for slides |

## 4.1 A report in one line

The function returns the ratios frame by default and writes the PDF to
`export_directory`.

One default to know about: `export_in_compacted_file='zip'`, so outputs are normally
bundled into an archive. Set it to `''` while exploring, to get the PDF on its own.

In [ ]:
report = rs.ratios_report(sample_df,
                          assessment_header=ASSESSMENT,
                          market_value_header=MARKET_VALUE,
                          group_by='property_type',
                          descriptive=True,
                          model_name='DemoModel',
                          export_directory=results_directory,
                          export_in_compacted_file='',
                          return_dict=False)
report

In [ ]:
# The PDF that call just produced. Note that model_name is slugged into the filename.
sorted(p.name for p in results_directory.glob('*DemoModel*.pdf'))

## 4.2 Personalising the report

`pdf_args` controls the document: opening and closing commentary, which chart families to
build, which characteristics to plot the ASR against, and display aliases so the reader
sees *"Living area (sq ft)"* rather than `living_area`.

The opening comments are the natural home for the methodology narrative the Standard calls for.

In [ ]:
pdf_args = {
    'pdf_starting_comments': [
        'Purpose: annual quality-assurance ratio study for the 2026 assessment cycle.',
        'Sales period: 44 months, time-adjusted to the 1 January 2026 valuation date.',
        'Screening: arm\'s-length sales only, per the Standard on Verification and '
        'Adjustment of Sales (IAAO 2020). No outlier trimming applied.',
        'Standards applied: level 0.90-1.10; COD 5.0-15.0 (residential improved, '
        'Table 11.1); VEI -10% to +10%.',
    ],
    'pdf_ending_comments': [
        'Prepared with the ratio_study package.',
        'Municipal Property Assessment Corporation - ASMA Data Science.',
    ],

    # Columns for the PDF summary tables. These must be real result columns:
    # 'VEM' is not one - the VEM reports as Inequity_conclusion / _severity /
    # _prevalence - and naming it here raises inside the PDF builder.
    'pdf_ratios_table_headers': ['Median', 'CoD', 'VEI', 'PRB', 'PRD'],

    # Chart families
    'pdf_overall_asr_graphs': True,
    'pdf_asr_graphs_x_vars_and_aliases': {
        'overall': {'living_area':   'Living area (sq ft)',
                    'quality_class': 'Quality class',
                    'sale_month':    'Month of sale'},
        'overall_histogram': True,
        'groups': [], 'groups_mosaic': [],
    },
    'pdf_grouping_aliases': {'property_type': 'Property type',
                             'neighbourhood': 'Economic neighbourhood'},

    # The vertical equity panel from section 6, per group as well as overall
    'pdf_vertical_equity_panel': True,
    'pdf_vertical_equity_panel_groups': True,
    'pdf_vertical_equity_panel_min_count': 30,
}

branded_report = rs.ratios_report(sample_df,
                                  assessment_header=ASSESSMENT,
                                  market_value_header=MARKET_VALUE,
                                  group_by=['property_type', 'neighbourhood'],
                                  descriptive=True,
                                  model_name='DemoModel_Branded',
                                  base_date='20260101',
                                  export_directory=results_directory,
                                  export_in_compacted_file='',
                                  pdf_args=pdf_args,
                                  return_dict=False)
branded_report

### Branding the PDF: logo, watermark, page numbers, cover and index

Five further `pdf_args` settings prepare the report for distribution. All are **optional and off by default**; each accepts a file path or `True` for the default settings, or a dictionary for finer control.

| Key | Adds | Options |
|---|---|---|
| `pdf_logo` | A logo on the report pages | `pages`: `'first'` (default) or `'all'`; `position`: `'top_left'` (default), `'top_center'`, `'top_right'`; `max_height`, `max_width` |
| `pdf_watermark` | An image behind the content | `position`: `'fill'` (default; as large as the page allows, never distorted), `'center'`, or a corner (`'top_left'`, `'top_right'`, `'bottom_left'`, `'bottom_right'`); `opacity` 0-1 (default 0.15); `scale` for positions other than fill |
| `pdf_page_numbers` | Page numbers | `position`: `'bottom_left'`, `'bottom_center'` (default), `'bottom_right'`; `font_size`; `format`: `'number'` or `'page_of_n'` ("Page 3 of 12") |
| `pdf_cover` | A cover page | `title`, `subtitle`, `authors`, `date` (default: today), and its own `logo` (independent of `pdf_logo`) |
| `pdf_index` | A title and contents page after the cover (requires `pdf_cover`) | `title` to override the cover title |

The cover page sits outside the body of the report: it has no page number or watermark and is not counted. The index page is also front matter, so the first report page is page 1, and the index refers to that numbering.

`report_title` replaces the default title (base date, "Sales Ratio Study" and `model_name`) with the exact text given. The cover uses it too, unless the cover sets its own `title`.

In [ ]:
# A placeholder logo for the demonstration - use your own image file in practice.
import matplotlib.pyplot as plt

logo_png = results_directory / 'demo_logo.png'
fig, ax = plt.subplots(figsize=(3, 1), dpi=200)
ax.text(0.5, 0.5, 'DEMO', ha='center', va='center', fontsize=36,
        fontweight='bold', color='#1f4e79')
ax.axis('off')
fig.savefig(logo_png, transparent=True, bbox_inches='tight')
plt.close(fig)

dressed_report = rs.ratios_report(sample_df,
                                  assessment_header=ASSESSMENT,
                                  market_value_header=MARKET_VALUE,
                                  group_by='property_type',
                                  model_name='DemoModel_Dressed',
                                  report_title='Residential Ratio Study - DemoModel',
                                  export_directory=results_directory,
                                  export_in_compacted_file='',
                                  pdf_args={
                                      'pdf_logo': {'path': logo_png, 'pages': 'all', 'position': 'top_right'},
                                      'pdf_watermark': {'path': logo_png, 'position': 'fill', 'opacity': 0.08},
                                      'pdf_page_numbers': {'position': 'bottom_right', 'format': 'page_of_n', 'font_size': 9},
                                      'pdf_cover': {'subtitle': 'Demonstration model',
                                                    'authors': ['A. Appraiser', 'B. Analyst'],
                                                    'logo': logo_png},
                                      'pdf_index': True,
                                  },
                                  return_dict=False)
dressed_report

## 4.3 Exporting the numbers too

`export_ratios` adds the data files, and `export_in_compacted_file` bundles the PDF and the
data into a single archive — the shape you want when a report has to be emailed or attached
to a ticket.

In [ ]:
exports = rs.ratios_report(sample_df,
                           assessment_header=ASSESSMENT,
                           market_value_header=MARKET_VALUE,
                           group_by=['property_type', 'neighbourhood'],
                           descriptive=True,
                           model_name='DemoModel_Bundle',
                           export_directory=results_directory,
                           export_ratios='both',              # xlsx + csv
                           export_in_compacted_file='zip',    # ... plus the PDF, zipped
                           return_dict=False)

archive = max(results_directory.glob('ratios_DemoModelBundle_*.zip'),
              key=lambda p: p.stat().st_mtime)
print('archive:', archive.name)

In [ ]:
import zipfile

with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        print(f'{info.file_size:>10,}  {info.filename}')

## 4.4 PNG output for slides

Any chart function writes a PNG when given `graph_filepath`, and returns the path.

## 4.4 PNG output for slides

Any chart function writes a PNG when given `graph_filepath`, and returns the path.

In [ ]:
panel_png = rs.vertical_equity_panel(
    sample_df,
    assessment_header=ASSESSMENT,
    market_value_header=MARKET_VALUE,
    title='Vertical Equity Panel',
    subtitle='DemoModel - all sales',
    graph_filepath=str(results_directory / 've_panel.png'))

print('saved:', panel_png)

In [ ]:
from IPython.display import Image, display
display(Image(filename=panel_png, width=620))

---
**Next:** [04 Advanced diagnostics](04_advanced_diagnostics.ipynb) shows the ML page of the report in detail.